# C01 — Modelos de Linguagem e Hugging Face

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)

**Autor:** Anderson Corrêa

**Projeto:** Letra da Lei — RAG hierarquia- e vigência-aware sobre o microssistema
penal federal brasileiro

Este notebook cobre o **ponto 1 da rubrica** (modelos de linguagem + Hugging Face).

Este notebook usa a biblioteca `transformers` (Hugging Face) diretamente sobre texto real do corpus jurídico
para expor conceitos fundamentais de PLN que sustentam as escolhas de arquitetura do
projeto: o *encoder* `intfloat/multilingual-e5-base` usado para recuperação
(`E5Embedder`) e o *decoder* `llama3.1:8b` (via Ollama) usado para geração (`OllamaClient`).

O que será demonstrado:

1. **Tokenização subword**: BERTimbau sobre o Art. 121 do Código Penal
2. **Encoder vs. decoder**: bidirecional vs. autorregressivo, e por que este projeto usa os dois
3. **PLN na prática**: similaridade semântica entre dispositivos (bi-encoder e5) e preenchimento de máscara (BERTimbau MLM)


In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS

corpus = load_corpus("data/raw", specs=[NORMS["CP"]])
cp = corpus.norm("CP")
art121 = cp.article("121")
print(art121.citation, "|", repr(art121.caput))


CP art. 121 | 'Matar alguem:'


## 1. Tokenização subword em português jurídico

[BERTimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased) é um BERT
pré-treinado do zero em português brasileiro (córpus brWaC). Como todo BERT, ele usa
**WordPiece**: um vocabulário fixo de ~30 mil subwords, onde palavras fora do vocabulário
são quebradas em pedaços menores (prefixados com `##` quando não iniciam palavra) em vez de
virarem um único token `[UNK]`. Vamos tokenizar o *caput* do Art. 121 do Código Penal
("Matar alguém") direto do corpus carregado acima.

Obs: O texto extraído do Planalto grafa "alguem" sem acento


In [2]:
from transformers import AutoTokenizer

TOKENIZER_NAME = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

sentence = f"{art121.caput} Pena - reclusão, de seis a vinte anos."
encoded = tokenizer(sentence)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

print("Frase:", sentence)
print(f"\n{len(tokens)} tokens (incl. [CLS]/[SEP]):")
for tok, id_ in zip(tokens, encoded["input_ids"]):
    print(f"  {tok!r:20s} -> {id_}")


Frase: Matar alguem: Pena - reclusão, de seis a vinte anos.

18 tokens (incl. [CLS]/[SEP]):
  '[CLS]'              -> 101
  'Mata'               -> 8372
  '##r'                -> 22282
  'algu'               -> 614
  '##em'               -> 210
  ':'                  -> 131
  'Pena'               -> 14241
  '-'                  -> 118
  'rec'                -> 570
  '##lusão'            -> 20323
  ','                  -> 117
  'de'                 -> 125
  'seis'               -> 2139
  'a'                  -> 123
  'vinte'              -> 4698
  'anos'               -> 481
  '.'                  -> 119
  '[SEP]'              -> 102


**Leitura:** `[CLS]` e `[SEP]` são tokens especiais que o BERT usa para marcar,
respectivamente, o início da sequência (cuja representação final é tipicamente usada para
classificação) e a fronteira/fim de sentença. Mesmo nesta frase curta e comum, duas palavras
já saem fragmentadas: "Matar" vira `Mata` + `##r`, e "reclusão" vira `rec` + `##lusão` — o
`##` marca continuação da palavra anterior, não um novo token independente. Palavras mais
frequentes como "Pena", "seis", "vinte" e "anos" permanecem inteiras.


In [3]:
terms = [
    "matar",           # common verb
    "prevaricação",    # rare legal jargon (CP art. 319)
    "concussão",       # rare legal jargon (CP art. 316)
    "estelionato",     # legal jargon (CP art. 171)
    "peculato",        # legal jargon (CP art. 312)
    "casa",            # common noun, control
]

print(f"{'termo':16s} {'nº subtokens':>12s}  subtokens")
for term in terms:
    sub = tokenizer.tokenize(term)
    print(f"{term:16s} {len(sub):>12d}  {sub}")


termo            nº subtokens  subtokens
matar                       1  ['matar']
prevaricação                3  ['prev', '##ari', '##cação']
concussão                   2  ['conc', '##ussão']
estelionato                 4  ['este', '##lio', '##na', '##to']
peculato                    2  ['pecul', '##ato']
casa                        1  ['casa']


**Leitura:** palavras comuns em minúsculas ("matar", "casa") viram um único token; termos
jurídicos mais raros e tecnicamente específicos ("prevaricação", "concussão", "estelionato",
"peculato" — nomes de crimes contra a administração pública/patrimônio pouco frequentes fora
de textos legais) são quebrados em mais subwords. Isso é o sintoma esperado de um vocabulário
WordPiece treinado num córpus **geral** (brWaC), não jurídico: termos raros no
pré-treinamento não ganham um token próprio. Note que "matar" vira 1 token, mas
"Matar" 2; BERTimbau é *cased*, então o vocabulário trata maiúsculas/minúsculas como entradas distintas, e nem toda variação de
capitalização de uma palavra comum tem seu próprio token.

**Implicação para este projeto:** um tokenizador/encoder genérico consegue *representar*
os termos, mas a granularidade fragmentada é evidência de que o modelo sabe pouco sobre eles.


## 2. Encoder vs. decoder

Os dois papéis são bem distintos:

- **Encoder (BERT/BERTimbau):** *self-attention* **bidirecional** — cada token atende à
  sequência inteira, à esquerda e à direita. Pré-treinado com *masked language modeling*. Produz representações
  contextuais ricas por token/sequência — ideal para **embeddings, similaridade e
  classificação**, mas não gera texto sequencialmente (não há noção de "próximo token").
- **Decoder (Llama/GPT):** *self-attention* **causal** — cada token só atende aos tokens
  anteriores (nunca ao futuro). Pré-treinado para prever o próximo token baseado nos antecessores. Ideal para **geração de texto**, mas suas representações intermediárias não
  são otimizadas para tarefas de similaridade como as de um encoder.

**Neste projeto usamos os dois, cada um no seu papel:**

| Papel | Modelo | Classe | Uso |
|---|---|---|---|
| Encoder | `intfloat/multilingual-e5-base` | `E5Embedder` (`direito_dados.retrieval.embedder`) | embeddings de consulta/dispositivo para recuperação vetorial (`VectorIndex`) |
| Decoder | `llama3.1:8b` (via Ollama) | `OllamaClient` (`direito_dados.generation.llm`) | geração da resposta citada (`answer_question`) e adjudicação de conflitos (`detect_conflicts`) |

A demonstração abaixo usa BERTimbau (fill-mask) para tornar a bidirecionalidade do encoder
concreta: mascaramos a **primeira** palavra de uma frase, de forma que a única maneira de
acertar é usando o contexto à **direita** — algo que um decoder autorregressivo (que só olha
para trás) não conseguiria fazer.


In [4]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=TOKENIZER_NAME, tokenizer=tokenizer)

# Mask on the FIRST word: getting it right requires using the context to the RIGHT
# ("capital da França"), which doesn't exist yet at the point where [MASK] is "read"
# left to right.
masked_sentence = f"{tokenizer.mask_token} é a capital da França."
for pred in fill_mask(masked_sentence)[:5]:
    print(f"{pred['score']:.3f}  {pred['sequence']}")


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.850  Paris é a capital da França.
0.051  Cannes é a capital da França.
0.045  Nice é a capital da França.
0.019  Lyon é a capital da França.
0.007  Bruxelas é a capital da França.


**Leitura:** o BERTimbau acerta "Paris" com alta confiança mesmo com a máscara na posição
0 — a única informação disponível está à **direita** ("é a capital da França").
Um decoder autorregressivo não conseguiria fazer essa previsão a partir do início vazio da sequência. Essa é exatamente a
propriedade (atenção bidirecional) que faz de um encoder um bom extrator de representação
para uma sequência inteira (embeddings), e que o torna, ao mesmo tempo, inadequado para
geração token a token porque sempre "vê" a frase inteira de uma só vez.


In [5]:
import inspect

from direito_dados.generation.llm import LLMClient, OllamaClient

llm = OllamaClient(model="llama3.1:8b")
print("Decoder usado na geração do projeto:", llm.model)
print("Assinatura de LLMClient.generate:", inspect.signature(OllamaClient.generate))
print("OllamaClient é compatível com o protocolo LLMClient:", isinstance(llm, LLMClient))


Decoder usado na geração do projeto: llama3.1:8b
Assinatura de LLMClient.generate: (self, prompt: str, system: str | None = None, format=None) -> str
OllamaClient é compatível com o protocolo LLMClient: True


`OllamaClient` só guarda configuração no `__init__` — nenhuma chamada de rede acontece até
`.generate()` ser invocado, então instanciá-lo aqui não depende do Ollama estar rodando.
Diferente de BERTimbau, `llama3.1:8b` gera token a token: cada novo token depende de todo o
prompt e de todos os tokens já gerados, por isso ele *pode*
continuar uma frase incompleta, mas *não poderia* fazer o preenchimento acima
usando o contexto à direita. A geração local está em `c04_inferencia_local_ou_remota.ipynb`. O presente notebook é focado em
Hugging Face/encoder e não depende do Ollama.

## 3. PLN: comparação entre dois modelos

Duas abordagens de PLN sobre o mesmo corpus, usando dois modelos diferentes:

- **Tarefa A — similaridade semântica entre dispositivos**, com o bi-encoder
  `multilingual-e5-base`, o encoder que este projeto usa para recuperação.
- **Tarefa B — preenchimento de máscara (cloze)**, com BERTimbau, aplicado a uma frase que exige
  vocabulário jurídico específico.


In [6]:
import numpy as np

from direito_dados.retrieval.embedder import E5Embedder


def excerpt(article, cap=200):
    return f"{article.caput} {article.text}"[:cap]


art155 = cp.article("155")  # theft (furto)
art157 = cp.article("157")  # robbery (roubo)
pairs = {
    "CP:art121 (homicídio)": art121,
    "CP:art155 (furto)": art155,
    "CP:art157 (roubo)": art157,
}

embedder = E5Embedder()
texts = [excerpt(a) for a in pairs.values()]
vectors = np.array(embedder.embed_passages(texts))

labels = list(pairs.keys())
sim = vectors @ vectors.T
print(f"{'':28s}" + "".join(f"{l[:18]:>20s}" for l in labels))
for i, l in enumerate(labels):
    print(f"{l:28s}" + "".join(f"{sim[i, j]:20.3f}" for j in range(len(labels))))


                              CP:art121 (homicíd   CP:art155 (furto)   CP:art157 (roubo)
CP:art121 (homicídio)                      1.000               0.906               0.858
CP:art155 (furto)                          0.906               1.000               0.920
CP:art157 (roubo)                          0.858               0.920               1.000


**Leitura:** furto (Art. 155, "subtrair coisa alheia móvel") e roubo (Art. 157, "subtrair
[...] mediante grave ameaça") são ambos crimes contra o **patrimônio**. O par
furto/roubo tem a maior similaridade da matriz (≈0.92), acima de homicídio×furto
(≈0.91) e acima de homicídio×roubo (≈0.86). A margem entre
furto×roubo e homicídio×furto é mais estreita do que se poderia esperar — os três artigos
compartilham vocabulário jurídico-penal genérico (pena de reclusão, sujeito, verbo no
infinitivo) no trecho de 200 caracteres usado aqui, o que empurra todas as similaridades para
cima. A propriedade de separar dispositivos mais relacionados dos menos relacionados no espaço vetorial é o que justifica o uso de `VectorIndex.query`.


In [7]:
theft_sentence = (
    f"Subtrair, para si ou para outrem, coisa alheia {tokenizer.mask_token}: "
    "pena de reclusão."
)
for pred in fill_mask(theft_sentence)[:5]:
    print(f"{pred['score']:.3f}  {pred['token_str']!r:15s} {pred['sequence']}")


0.082  'proibida'      Subtrair, para si ou para outrem, coisa alheia proibida : pena de reclusão.
0.055  'qualquer'      Subtrair, para si ou para outrem, coisa alheia qualquer : pena de reclusão.
0.053  '.'             Subtrair, para si ou para outrem, coisa alheia. : pena de reclusão.
0.049  'a'             Subtrair, para si ou para outrem, coisa alheia a : pena de reclusão.
0.039  '"'             Subtrair, para si ou para outrem, coisa alheia " : pena de reclusão.


**Leitura:** o *caput* real do Art. 155 é "Subtrair, para si ou para outrem, coisa alheia
**móvel**" — e `móvel` **não** aparece entre as top-5 previsões acima, cujos scores (todos
abaixo de 0.1) são muito mais baixos que os 0.850 obtidos no cloze genérico da Seção 2
("Paris"). O modelo captura corretamente o **padrão sintático** (todas as previsões são
adjetivos/pronomes plausíveis na posição "coisa alheia ___"), mas erra o termo técnico
específico — evidência direta de que o BERTimbau, pré-treinado num córpus geral, **não**
memorizou de forma confiável a colocação jurídica exata "coisa alheia móvel" (o elemento que
distingue furto de outros crimes patrimoniais no Código Penal).

Isso reforça o argumento da Seção 1 (tokenização/OOV): a arquitetura bidirecional funciona perfeitamente bem (como mostrou o exemplo "Paris"), mas o
**conhecimento paramétrico** de um encoder geral sobre vocabulário jurídico específico é raso e não confiável, motivo pelo qual este projeto **recupera o texto exato do artigo** antes de gerar qualquer resposta (RAG, `c05_rag_pipeline.ipynb`).

### e5 (bi-encoder) vs. BERTimbau (MLM) — por que o projeto usa um e não o outro para recuperação

| | `multilingual-e5-base` | BERTimbau |
|---|---|---|
| Tipo | encoder treinado especificamente para *sentence embeddings* (contrastive learning, prefixos `query:`/`passage:` assimétricos) | encoder de propósito geral, pré-treinado só com MLM/NSP |
| Saída usada aqui | vetor de sentença normalizado, pronto para similaridade por cosseno | logits por token mascarado (cloze), não uma representação de sentença |
| Uso neste projeto | `E5Embedder`, recuperação em produção (`VectorIndex`) | nenhum — usado só nesta notebook para ilustrar conceitos |
| Por quê | e5 foi treinado para que a similaridade de embeddings *já* reflita similaridade semântica; usar o `[CLS]` cru do BERTimbau para isso exigiria fine-tuning adicional (ex.: Sentence-BERT) para produzir embeddings de qualidade comparável | BERTimbau sem esse ajuste tende a produzir embeddings anisotrópicos, piores para ranquear por similaridade "de fábrica" |

Os dois são encoders — a diferença é o **objetivo de treinamento**: e5 é otimizado para embeddings comparáveis por similaridade;
BERTimbau é otimizado para prever tokens mascarados, uma tarefa relacionada mas não idêntica.


## Conclusão

Três resultados explicam as escolhas de arquitetura do projeto:

1. **Tokenização subword**: vocabulário jurídico
   ("prevaricação", "concussão", "peculato"...) é fragmentado em mais subtokens que
   vocabulário comum, sendo isso um argumento a favor de usar RAG.
2. **Encoder vs. decoder**: o projeto usa `E5Embedder` (encoder) para que a recuperação funcione por similaridade vetorial, e
   `OllamaClient`/`llama3.1:8b` (decoder) para gerar a resposta em linguagem natural a partir do que foi recuperado.

A geração local é coberta em `c04_inferencia_local_ou_remota.ipynb`.
